In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    initcap,
    lower,
    upper,
    regexp_replace,
    when,
    count,
    sum as spark_sum
)

spark = SparkSession.builder \
    .appName("CustomerDataCleansing") \
    .master("local[*]") \
    .getOrCreate()
print("Spark Session created successfully.")
spark.sparkContext.setLogLevel("WARN")

In [ ]:
df = spark.read.csv(
    "customer_data.csv",
    header=True,
    inferSchema=True
)

print("\n===== RAW CUSTOMER DATA =====")
df.show(truncate=False)

print("\n===== SCHEMA =====")
df.printSchema()

In [ ]:
raw_count = df.count()

print("Total Raw Records:", raw_count)

In [ ]:
print("\n===== NULL VALUE REPORT =====")

df.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in df.columns
]).show()

In [ ]:
print("\n===== DUPLICATE CUSTOMER IDs =====")

duplicates = df.groupBy("CustomerID") \
    .count() \
    .filter(col("count") > 1)

duplicates.show()

In [ ]:
duplicate_count = df.count() - \
                  df.dropDuplicates(["CustomerID"]).count()

print("Duplicate Records:", duplicate_count)

In [ ]:
invalid_age_count = df.filter(
    col("Age").isNotNull() &
    ((col("Age") < 0) | (col("Age") > 120))
).count()

print("Invalid Ages:", invalid_age_count)

In [ ]:
email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

invalid_email_count = df.filter(
    col("Email").isNull() |
    (~col("Email").rlike(email_pattern))
).count()

print("Invalid Emails:", invalid_email_count)

In [ ]:
clean_df = df.withColumn(
    "Name",
    initcap(trim(col("Name")))
)
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.withColumn(
    "City",
    initcap(trim(col("City")))
)
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.na.fill({
    "City": "Unknown"
})
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.withColumn(
    "Phone",
    when(
        col("Phone").rlike("^[0-9]{10}$"),
        col("Phone")
    ).otherwise(None)
)
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.withColumn(
    "Email",
    lower(trim(col("Email")))
)
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.withColumn(
    "Email",
    when(
        col("Email").rlike(email_pattern),
        col("Email")
    ).otherwise(None)
)
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.withColumn(
    "Age",
    when(
        (col("Age") >= 0) &
        (col("Age") <= 120),
        col("Age")
    ).otherwise(None)
)
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.withColumn(
    "Gender",
    when(
        upper(trim(col("Gender"))).isin("M", "MALE"),
        "Male"
    ).when(
        upper(trim(col("Gender"))).isin("F", "FEMALE"),
        "Female"
    ).otherwise("Unknown")
)
clean_df.show(truncate=False)

In [ ]:
clean_df = clean_df.dropDuplicates(
    ["CustomerID"]
)
clean_df.show(truncate=False)

In [ ]:
print("\n===== CLEANED CUSTOMER DATA =====")

clean_df.orderBy("CustomerID").show(
    truncate=False
)

In [ ]:
remaining_duplicates = (
    clean_df.count()
    - clean_df.dropDuplicates(["CustomerID"]).count()
)

remaining_invalid_ages = clean_df.filter(
    col("Age").isNotNull() &
    ((col("Age") < 0) | (col("Age") > 120))
).count()

remaining_invalid_emails = clean_df.filter(
    col("Email").isNotNull() &
    (~col("Email").rlike(email_pattern))
).count()

remaining_invalid_phones = clean_df.filter(
    col("Phone").isNotNull() &
    (~col("Phone").rlike("^[0-9]{10}$"))
).count()

clean_count = clean_df.count()

validation_data = [
    (
        "Total Raw Records",
        str(raw_count),
        str(clean_count),
        "INFO"
    ),
    (
        "Duplicate Customer IDs",
        str(duplicate_count),
        str(remaining_duplicates),
        "PASS" if remaining_duplicates == 0 else "FAIL"
    ),
    (
        "Invalid Ages",
        str(invalid_age_count),
        str(remaining_invalid_ages),
        "PASS" if remaining_invalid_ages == 0 else "FAIL"
    ),
    (
        "Invalid Email Formats",
        str(invalid_email_count),
        str(remaining_invalid_emails),
        "PASS" if remaining_invalid_emails == 0 else "FAIL"
    ),
    (
        "Invalid Phone Formats",
        "Not checked before normalization",
        str(remaining_invalid_phones),
        "PASS" if remaining_invalid_phones == 0 else "FAIL"
    )
]

validation_df = spark.createDataFrame(
    validation_data,
    [
        "ValidationCheck",
        "BeforeCleaning",
        "AfterCleaning",
        "Status"
    ]
)

print("\n===== VALIDATION REPORT =====")

validation_df.show(
    truncate=False
)

In [ ]:
print("\n===== NULL REPORT AFTER CLEANING =====")

null_report = clean_df.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in clean_df.columns
])

null_report.show()

In [23]:
clean_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/cleaned_customers")

In [24]:
clean_df.write \
    .mode("overwrite") \
    .parquet("output/cleaned_customers_parquet")

In [25]:
validation_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/validation_report")